# Baseline models for chiller schedule reconstruction

**Logistic regression · Histogram gradient boosting · Frozen MLP**

This notebook compares two conventional classifiers with the original five-seed neural-network ensemble. All three methods use the same 41 contextual features, seven leave-one-system-out splits, fixed classification threshold, and daily-event decoder.

The neural network is **not retrained**: its existing predictions, feature tables, and reference states are read from an authorized local experiment directory. No measurements or results are distributed with this notebook.

In [ ]:
# Configuration and dependencies
import json
import math
import os
import time
import warnings
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    f1_score, log_loss, matthews_corrcoef, precision_score,
    recall_score, roc_auc_score,
)
from sklearn.preprocessing import RobustScaler

# In Colab, mount Drive to access the private results directory.
try:
    from google.colab import drive
except ImportError:
    pass
else:
    drive.mount('/content/drive')

# Set TASK1_RESULTS_DIR to the folder containing the original run manifest
# and its features/, truth/, predictions/, and tables/ subdirectories.
# Example: os.environ['TASK1_RESULTS_DIR'] = '/content/drive/MyDrive/your-private-results'
RESULT_DIR = Path(os.environ.get('TASK1_RESULTS_DIR', 'results')).expanduser()
FEATURE_DIR = RESULT_DIR / 'features'
TRUTH_DIR = RESULT_DIR / 'truth'
PRED_DIR = RESULT_DIR / 'predictions'
TABLE_DIR = RESULT_DIR / 'tables'
COMPARISON_DIR = RESULT_DIR / 'baseline_comparison'
MANIFEST_FILE = RESULT_DIR / 'Run_Manifest_FINAL_PAPER.json'

MAX_FOLDS = None  # All seven LOSO folds.
RUN_PAIRED_BOOTSTRAP = True
BOOTSTRAP_REPLICATES = 2000
BOOTSTRAP_SEED = 20261009
SAVE_INTERVAL_PREDICTIONS = False

# Validation-system hyperparameter selection; test-system labels are never used.
LOGISTIC_GRID = [0.1, 1.0, 10.0]
BOOSTING_GRID = [
    dict(max_iter=150, max_leaf_nodes=15, learning_rate=0.10, l2_regularization=1.0),
    dict(max_iter=200, max_leaf_nodes=31, learning_rate=0.10, l2_regularization=1.0),
    dict(max_iter=300, max_leaf_nodes=31, learning_rate=0.05, l2_regularization=1.0),
]

required = [MANIFEST_FILE, FEATURE_DIR, TRUTH_DIR, PRED_DIR, TABLE_DIR]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Missing cached experiment outputs. Set TASK1_RESULTS_DIR to the original '
        'results directory. Missing: ' + '; '.join(missing)
    )
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
manifest = json.loads(MANIFEST_FILE.read_text(encoding='utf-8'))
BUILDING_NAMES = list(manifest['building_set'])
FEATURE_NAMES = list(manifest['feature_names'])
assert len(BUILDING_NAMES) == 7 and len(set(BUILDING_NAMES)) == 7
assert len(FEATURE_NAMES) == 41 and len(set(FEATURE_NAMES)) == 41
PUBLIC_CODE = {name: f'B{i + 1}' for i, name in enumerate(BUILDING_NAMES)}

# Event and evaluation settings from the original experiment.
CFG = SimpleNamespace(
    dt_minutes=15, decision_threshold=0.50,
    event_pre_window_min=60, event_pre_dominance_ratio=0.75,
    event_pre_min_valid_fraction=0.75, event_post_steps=2,
    morning_start='04:00', morning_end='10:00',
    day_status_min_coverage=0.90, min_truth_day_coverage=0.90,
    min_month_interval_coverage=0.90, min_month_truth_coverage=0.90,
    timing_tolerance_15_min=15, timing_tolerance_30_min=30,
)
print(f'Loaded run configuration: {len(BUILDING_NAMES)} systems, {len(FEATURE_NAMES)} features.')

## Shared evaluation functions

The following functions implement the original event definitions, training weights, fold splits, schedule summaries, and performance measures. Their computational definitions are retained from the baseline experiment.

In [ ]:
# 2. Authoritative helper definitions from the original notebook (unmodified).

def season_mask(index):
    mask = np.zeros(len(index), dtype=bool)
    for start, end in SEASONS.values():
        mask |= (index >= start) & (index <= end)
    return mask

def regularize_binary_state(state):
    s = pd.to_numeric(state, errors='coerce').astype(float).sort_index()
    if s.empty:
        return s
    s = s.loc[~s.index.duplicated(keep='last')]
    full_idx = pd.date_range(s.index.min(), s.index.max(), freq=f'{CFG.dt_minutes}min')
    return s.reindex(full_idx)

def vectorized_transition_masks(state):
    s = regularize_binary_state(state)
    if s.empty:
        empty = pd.Series(False, index=s.index, dtype=bool)
        return s, empty, empty.copy(), empty.copy()

    pre_steps = int(CFG.event_pre_window_min / CFG.dt_minutes)
    min_valid = int(math.ceil(pre_steps * CFG.event_pre_min_valid_fraction))
    lagged = s.shift(1)

    pre_valid_n = lagged.notna().astype(np.int8).rolling(pre_steps, min_periods=1).sum()
    pre_off_n = lagged.eq(0).astype(np.int8).rolling(pre_steps, min_periods=1).sum()
    pre_on_n = lagged.eq(1).astype(np.int8).rolling(pre_steps, min_periods=1).sum()
    denom = pre_valid_n.replace(0, np.nan)

    pre_off_ratio = pre_off_n / denom
    pre_on_ratio = pre_on_n / denom
    prev = s.shift(1)

    post_on = pd.Series(True, index=s.index, dtype=bool)
    post_off = pd.Series(True, index=s.index, dtype=bool)
    for j in range(CFG.event_post_steps):
        future = s.shift(-j)
        post_on &= future.eq(1)
        post_off &= future.eq(0)

    enough_pre = pre_valid_n >= min_valid

    valid_on = (
        enough_pre
        & (pre_off_ratio >= CFG.event_pre_dominance_ratio)
        & prev.eq(0)
        & post_on
    ).fillna(False)

    valid_off = (
        enough_pre
        & (pre_on_ratio >= CFG.event_pre_dominance_ratio)
        & prev.eq(1)
        & post_off
    ).fillna(False)

    raw_transition = (
        s.notna()
        & prev.notna()
        & s.ne(prev)
    ).fillna(False)

    return s, valid_on.astype(bool), valid_off.astype(bool), raw_transition.astype(bool)

def _sustained_state_at(regular_state, ts, required_state):
    ts = pd.Timestamp(ts)
    dt = pd.Timedelta(minutes=CFG.dt_minutes)
    pre_steps = int(CFG.event_pre_window_min / CFG.dt_minutes)

    pre_idx = pd.date_range(
        ts - pre_steps * dt,
        ts - dt,
        freq=f'{CFG.dt_minutes}min',
    )
    pre = regular_state.reindex(pre_idx)
    n_valid = int(pre.notna().sum())
    min_valid = int(math.ceil(pre_steps * CFG.event_pre_min_valid_fraction))
    if n_valid < min_valid:
        return False

    dominance = float((pre.dropna() == required_state).mean())
    if dominance < CFG.event_pre_dominance_ratio:
        return False

    for j in range(CFG.event_post_steps):
        tj = ts + pd.Timedelta(minutes=CFG.dt_minutes * j)
        if (
            tj not in regular_state.index
            or pd.isna(regular_state.loc[tj])
            or int(regular_state.loc[tj]) != required_state
        ):
            return False
    return True

def build_daily_operational_table(state):
    s, valid_on_mask, valid_off_mask, raw_transition_mask = vectorized_transition_masks(state)
    if s.empty:
        return pd.DataFrame()

    on_times = valid_on_mask.index[valid_on_mask.to_numpy()]
    off_times = valid_off_mask.index[valid_off_mask.to_numpy()]
    raw_times = raw_transition_mask.index[raw_transition_mask.to_numpy()]

    on_by_day, off_by_day, raw_by_day = {}, {}, {}
    for ts in on_times:
        on_by_day.setdefault(ts.normalize(), []).append(ts)
    for ts in off_times:
        off_by_day.setdefault(ts.normalize(), []).append(ts)
    for ts in raw_times:
        raw_by_day.setdefault(ts.normalize(), []).append(ts)

    rows = []
    first_day = s.index.min().normalize()
    last_day = s.index.max().normalize()

    for date in pd.date_range(first_day, last_day, freq='D'):
        day_end = date + pd.Timedelta(hours=23, minutes=45)
        day_vals = s.loc[date:day_end]
        day_coverage = float(day_vals.notna().mean())
        valid_vals = day_vals.dropna()

        always_on = bool(
            day_coverage >= CFG.day_status_min_coverage
            and len(valid_vals)
            and valid_vals.eq(1).all()
        )
        always_off = bool(
            day_coverage >= CFG.day_status_min_coverage
            and len(valid_vals)
            and valid_vals.eq(0).all()
        )

        search_start = pd.Timestamp(f'{date.date()} {CFG.morning_start}')
        search_end = pd.Timestamp(f'{date.date()} {CFG.morning_end}')

        day_on = on_by_day.get(date, [])
        day_off = off_by_day.get(date, [])

        morning_on = [ts for ts in day_on if search_start <= ts <= search_end]
        on_transition = morning_on[0] if morning_on else pd.NaT

        # Retained left-edge reporting for start time.
        on_time = (
            on_transition - pd.Timedelta(minutes=CFG.dt_minutes)
            if pd.notna(on_transition)
            else pd.NaT
        )

        already_on_at_04 = _sustained_state_at(s, search_start, 1)

        # Principal shutdown semantics:
        #   * if a principal morning start exists, use the last validated OFF after it;
        #   * if the chiller was already ON at 04:00, use the last validated OFF after 04:00;
        #   * otherwise DO NOT assign a later-cycle shutdown as the principal daily shutdown.
        if pd.notna(on_transition):
            eligible_off = [ts for ts in day_off if ts > on_transition]
        elif already_on_at_04:
            eligible_off = [ts for ts in day_off if ts >= search_start]
        else:
            eligible_off = []

        off_time = eligible_off[-1] if eligible_off else pd.NaT

        if day_coverage < CFG.day_status_min_coverage:
            day_status = 'INSUFFICIENT_DATA'
        elif always_on:
            day_status = 'ALWAYS_ON'
        elif always_off:
            day_status = 'ALWAYS_OFF'
        elif pd.notna(on_time) and pd.notna(off_time):
            day_status = 'START_AND_STOP'
        elif pd.notna(on_time):
            day_status = 'START_ONLY'
        elif pd.notna(off_time):
            day_status = 'STOP_ONLY'
        elif already_on_at_04:
            day_status = 'ALREADY_ON_NO_VALID_STOP'
        else:
            day_status = 'MIXED_NO_PRIMARY_EVENT'

        rows.append({
            'date': date,
            'turn_on_transition_time': on_transition,
            'turn_on_time': on_time,
            'turn_off_time': off_time,
            'already_on_at_04': bool(already_on_at_04),
            'always_on': always_on,
            'always_off': always_off,
            'day_status': day_status,
            'day_state_coverage': day_coverage,
            'n_valid_on_candidates': int(len(day_on)),
            'n_valid_off_candidates': int(len(day_off)),
            'n_raw_transitions': int(len(raw_by_day.get(date, []))),
        })

    return pd.DataFrame(rows)

def supervised_rows(buildings):
    parts = []
    for b in buildings:
        df = FEATURE_TABLES[b]
        if len(df):
            valid = (
                df['true_state'].notna()
                & df['primary_event_distance'].notna()
                & df['secondary_transition_distance'].notna()
            )
            parts.append(df.loc[valid].copy())
    return pd.concat(parts).sort_index() if parts else pd.DataFrame()

def fit_fold_scaler(train_df):
    scaler = RobustScaler(
        quantile_range=(25, 75),
        with_centering=True,
        with_scaling=True,
        unit_variance=False,
    )
    scaler.fit(train_df[FEATURE_NAMES].to_numpy(float))
    scale = scaler.scale_.copy()
    scale[~np.isfinite(scale) | (np.abs(scale) < 1e-9)] = 1.0
    return scaler.center_.astype(np.float32), scale.astype(np.float32)

def balanced_sample_weights(frame):
    """Equal expected mass per building, then per month within building."""
    tmp = frame[['building', 'month_id']].reset_index(drop=True)
    months_per_building = tmp.groupby('building')['month_id'].nunique().to_dict()
    group_sizes = tmp.groupby(['building', 'month_id']).size().to_dict()

    w = np.empty(len(tmp), dtype=np.float64)
    for i, (b, m) in enumerate(zip(tmp['building'], tmp['month_id'])):
        w[i] = 1.0 / (months_per_building[b] * group_sizes[(b, m)])

    w /= np.sum(w)
    return w

def fold_split(test_building):
    i = BUILDING_NAMES.index(test_building)
    val_building = BUILDING_NAMES[(i + 1) % len(BUILDING_NAMES)]
    train_buildings = [
        b for b in BUILDING_NAMES
        if b not in {test_building, val_building}
    ]
    return train_buildings, val_building

def threshold_state(score_series, threshold=None):
    """Final fixed-threshold decoder."""
    if threshold is None:
        threshold = CFG.decision_threshold
    s = pd.to_numeric(score_series, errors='coerce')
    out = pd.Series(np.nan, index=s.index, dtype=float)
    m = s.notna()
    out.loc[m] = (s.loc[m] >= float(threshold)).astype(float)
    return out

def clean_isolated_binary_flips(state):
    s = pd.to_numeric(state, errors='coerce').astype(float).sort_index()
    if len(s) < 3:
        return s.copy()

    prev = s.shift(1)
    nxt = s.shift(-1)
    idx = s.index.to_series()

    contiguous_prev = idx.diff() == pd.Timedelta(minutes=CFG.dt_minutes)
    contiguous_next = (idx.shift(-1) - idx) == pd.Timedelta(minutes=CFG.dt_minutes)

    flip = (
        s.notna() & prev.notna() & nxt.notna()
        & contiguous_prev & contiguous_next
        & prev.eq(nxt) & s.ne(prev)
    )

    out = s.copy()
    out.loc[flip] = prev.loc[flip]
    return out

def time_to_minutes(ts):
    if pd.isna(ts):
        return np.nan
    ts = pd.Timestamp(ts)
    return float(ts.hour * 60 + ts.minute + ts.second / 60.0)

def minutes_to_hhmm(v):
    if not np.isfinite(v):
        return ''
    v = float(v) % 1440.0
    h = int(v // 60)
    m = int(round(v - h * 60))
    if m == 60:
        h = (h + 1) % 24
        m = 0
    return f'{h:02d}:{m:02d}'

def _metric_mask(mask, n):
    if mask is None:
        return np.ones(int(n), dtype=bool)
    m = pd.Series(mask).fillna(False).to_numpy(dtype=bool)
    if len(m) != int(n):
        raise ValueError(f'Evaluation-mask length {len(m)} != data length {n}')
    return m

def event_detection_stats(true_times, pred_times, prefix, eval_mask=None):
    t = pd.Series(true_times).notna().to_numpy()
    p = pd.Series(pred_times).notna().to_numpy()
    m = _metric_mask(eval_mask, len(t))
    t = t[m]
    p = p[m]

    tp = int(np.sum(t & p))
    fp = int(np.sum(~t & p))
    fn = int(np.sum(t & ~p))

    precision = tp / (tp + fp) if (tp + fp) else np.nan
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if np.isfinite(precision) and np.isfinite(recall) and (precision + recall) > 0
        else np.nan
    )

    return {
        f'{prefix}_scored_days': int(len(t)),
        f'{prefix}_tp_days': tp,
        f'{prefix}_false_days': fp,
        f'{prefix}_missed_days': fn,
        f'{prefix}_precision': precision,
        f'{prefix}_recall': recall,
        f'{prefix}_f1': f1,
    }

def matched_timing_stats(true_times, pred_times, prefix, eval_mask=None):
    t = pd.to_datetime(pd.Series(true_times), errors='coerce')
    p = pd.to_datetime(pd.Series(pred_times), errors='coerce')
    m = _metric_mask(eval_mask, len(t))
    both = t.notna() & p.notna() & m

    if not both.any():
        return {
            f'{prefix}_matched_n': 0,
            f'{prefix}_mae_min': np.nan,
            f'{prefix}_median_ae_min': np.nan,
            f'{prefix}_rmse_min': np.nan,
            f'{prefix}_bias_min': np.nan,
            f'{prefix}_q90_ae_min': np.nan,
            f'{prefix}_within15': np.nan,
            f'{prefix}_within30': np.nan,
        }

    err = (
        p.loc[both].reset_index(drop=True)
        - t.loc[both].reset_index(drop=True)
    ).dt.total_seconds() / 60.0
    ae = err.abs()

    return {
        f'{prefix}_matched_n': int(len(err)),
        f'{prefix}_mae_min': float(ae.mean()),
        f'{prefix}_median_ae_min': float(ae.median()),
        f'{prefix}_rmse_min': float(np.sqrt(np.mean(np.square(err)))),
        f'{prefix}_bias_min': float(err.mean()),
        f'{prefix}_q90_ae_min': float(ae.quantile(0.90)),
        f'{prefix}_within15': float((ae <= CFG.timing_tolerance_15_min).mean()),
        f'{prefix}_within30': float((ae <= CFG.timing_tolerance_30_min).mean()),
    }

def schedule_distribution_stats(series, prefix):
    vals = np.array(
        [time_to_minutes(x) for x in series if pd.notna(x)],
        dtype=float,
    )
    vals = vals[np.isfinite(vals)]

    out = {f'{prefix}_n': int(len(vals))}
    names = ['mean','std','q10','q25','median','q75','q90','iqr']

    if not len(vals):
        for name in names:
            out[f'{prefix}_{name}_min'] = np.nan
        for name in ['q10','q25','median','q75','q90']:
            out[f'{prefix}_{name}_hhmm'] = ''
        return out

    q10, q25, med, q75, q90 = np.quantile(vals, [0.10,0.25,0.50,0.75,0.90])
    out.update({
        f'{prefix}_mean_min': float(np.mean(vals)),
        f'{prefix}_std_min': float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0,
        f'{prefix}_q10_min': float(q10),
        f'{prefix}_q25_min': float(q25),
        f'{prefix}_median_min': float(med),
        f'{prefix}_q75_min': float(q75),
        f'{prefix}_q90_min': float(q90),
        f'{prefix}_iqr_min': float(q75 - q25),
        f'{prefix}_q10_hhmm': minutes_to_hhmm(q10),
        f'{prefix}_q25_hhmm': minutes_to_hhmm(q25),
        f'{prefix}_median_hhmm': minutes_to_hhmm(med),
        f'{prefix}_q75_hhmm': minutes_to_hhmm(q75),
        f'{prefix}_q90_hhmm': minutes_to_hhmm(q90),
    })
    return out

def month_expected_index(year, month):
    idx = load_raw.index[season_mask(load_raw.index)]
    mask = (idx.year == int(year)) & (idx.month == int(month))
    return idx[mask]

def month_qc(feature_month, truth_state, year, month):
    expected = month_expected_index(year, month)
    expected_n = int(len(expected))
    feature_valid_n = int(len(feature_month.index.unique()))
    feature_coverage = feature_valid_n / expected_n if expected_n else np.nan

    truth_reg = pd.to_numeric(truth_state, errors='coerce').reindex(expected)
    truth_valid_n = int(truth_reg.notna().sum())
    truth_coverage = truth_valid_n / expected_n if expected_n else np.nan

    m0 = pd.Timestamp(year=int(year), month=int(month), day=1)
    mend = m0 + pd.offsets.MonthEnd(1) + pd.Timedelta(hours=23, minutes=45)

    complete = bool(
        expected_n > 0
        and len(expected) > 0
        and expected.min() <= m0
        and expected.max() >= mend
    )
    coverage_eligible = bool(
        np.isfinite(feature_coverage)
        and np.isfinite(truth_coverage)
        and feature_coverage >= CFG.min_month_interval_coverage
        and truth_coverage >= CFG.min_month_truth_coverage
    )

    return {
        'expected_interval_n': expected_n,
        'feature_interval_n': feature_valid_n,
        'truth_interval_n': truth_valid_n,
        'feature_interval_coverage': feature_coverage,
        'truth_interval_coverage': truth_coverage,
        # Backward-compatible alias used by some figures/tables.
        'month_interval_coverage': feature_coverage,
        'complete_calendar_month': complete,
        'coverage_eligible': coverage_eligible,
        'evaluation_eligible': coverage_eligible,
        'primary_evaluation_eligible': bool(coverage_eligible and complete),
    }

def _analysis_dates_index():
    dates = []
    for season_start, season_end in SEASONS.values():
        dates.extend(
            pd.date_range(
                pd.Timestamp(season_start).normalize(),
                pd.Timestamp(season_end).normalize(),
                freq='D',
            )
        )
    return pd.DatetimeIndex(pd.unique(pd.DatetimeIndex(dates)))

def build_daily_comparison(true_state, pred_state):
    true_daily = build_daily_operational_table(true_state)
    pred_daily = build_daily_operational_table(pred_state)

    true_daily = true_daily.loc[
        pd.to_datetime(true_daily['date']).dt.normalize().isin(ANALYSIS_DATES)
    ].copy()
    pred_daily = pred_daily.loc[
        pd.to_datetime(pred_daily['date']).dt.normalize().isin(ANALYSIS_DATES)
    ].copy()

    true_daily = true_daily.rename(columns={
        'turn_on_time':'true_turn_on',
        'turn_off_time':'true_turn_off',
        'day_status':'true_day_status',
        'always_on':'true_always_on',
        'always_off':'true_always_off',
        'already_on_at_04':'true_already_on_at_04',
        'day_state_coverage':'true_day_state_coverage',
        'n_raw_transitions':'true_raw_transitions',
    })
    pred_daily = pred_daily.rename(columns={
        'turn_on_time':'pred_turn_on',
        'turn_off_time':'pred_turn_off',
        'day_status':'pred_day_status',
        'always_on':'pred_always_on',
        'always_off':'pred_always_off',
        'already_on_at_04':'pred_already_on_at_04',
        'day_state_coverage':'pred_day_state_coverage',
        'n_raw_transitions':'pred_raw_transitions',
    })

    daily = true_daily[[
        'date','true_turn_on','true_turn_off','true_day_status',
        'true_always_on','true_always_off','true_already_on_at_04',
        'true_day_state_coverage','true_raw_transitions',
    ]].merge(
        pred_daily[[
            'date','pred_turn_on','pred_turn_off','pred_day_status',
            'pred_always_on','pred_always_off','pred_already_on_at_04',
            'pred_day_state_coverage','pred_raw_transitions',
        ]],
        on='date',
        how='outer',
    ).sort_values('date').reset_index(drop=True)

    daily['daily_eval_eligible'] = (
        pd.to_numeric(daily['true_day_state_coverage'], errors='coerce')
        >= CFG.min_truth_day_coverage
    ) & daily['true_day_status'].ne('INSUFFICIENT_DATA')

    both_on = (
        daily['daily_eval_eligible']
        & daily['true_turn_on'].notna()
        & daily['pred_turn_on'].notna()
    )
    both_off = (
        daily['daily_eval_eligible']
        & daily['true_turn_off'].notna()
        & daily['pred_turn_off'].notna()
    )

    daily['turn_on_error_min'] = np.nan
    daily['turn_off_error_min'] = np.nan

    daily.loc[both_on, 'turn_on_error_min'] = (
        pd.to_datetime(daily.loc[both_on, 'pred_turn_on'])
        - pd.to_datetime(daily.loc[both_on, 'true_turn_on'])
    ).dt.total_seconds() / 60.0

    daily.loc[both_off, 'turn_off_error_min'] = (
        pd.to_datetime(daily.loc[both_off, 'pred_turn_off'])
        - pd.to_datetime(daily.loc[both_off, 'true_turn_off'])
    ).dt.total_seconds() / 60.0

    daily['turn_on_detection_match'] = np.where(
        daily['daily_eval_eligible'],
        daily['true_turn_on'].notna() == daily['pred_turn_on'].notna(),
        np.nan,
    )
    daily['turn_off_detection_match'] = np.where(
        daily['daily_eval_eligible'],
        daily['true_turn_off'].notna() == daily['pred_turn_off'].notna(),
        np.nan,
    )
    daily['always_on_match'] = np.where(
        daily['daily_eval_eligible'],
        daily['true_always_on'] == daily['pred_always_on'],
        np.nan,
    )
    daily['always_off_match'] = np.where(
        daily['daily_eval_eligible'],
        daily['true_always_off'] == daily['pred_always_off'],
        np.nan,
    )
    return daily

## Frozen experiment inputs

Read the original contextual features, reference labels, and five-seed MLP predictions. Check alignment and verify that fixed-threshold decoding reproduces the saved MLP states.

In [ ]:
# 3. Reuse cached tables created by the original analysis.
FEATURE_TABLES, TRUTH, MLP_INTERVALS = {}, {}, {}
for system in BUILDING_NAMES:
    feature_path = FEATURE_DIR / f'{system}_context_features.pkl.gz'
    truth_path = TRUTH_DIR / f'{system}_truth.pkl.gz'
    prediction_path = PRED_DIR / f'{system}_interval_predictions.csv'
    for path in [feature_path, truth_path, prediction_path]:
        if not path.is_file():
            raise FileNotFoundError(f'Missing original result: {path}')
    feature = pd.read_pickle(feature_path, compression='gzip')
    truth = pd.read_pickle(truth_path, compression='gzip')
    mlp = pd.read_csv(prediction_path, index_col=0, parse_dates=True)
    assert feature.index.is_unique and mlp.index.is_unique
    assert feature.index.equals(mlp.index), 'Original features/predictions index mismatch'
    assert all(name in feature.columns for name in FEATURE_NAMES)
    assert np.isfinite(feature[FEATURE_NAMES].to_numpy(dtype=float)).all()
    pd.testing.assert_series_equal(
        feature['true_state'].fillna(-999).astype(float),
        mlp['true_state'].fillna(-999).astype(float), check_names=False,
    )
    FEATURE_TABLES[system] = feature
    TRUTH[system] = {'work': truth}
    MLP_INTERVALS[system] = mlp

# Follow the original calendar index and cooling-season coverage implicit in its truth caches.
reference_grid = TRUTH[BUILDING_NAMES[0]]['work'].index
assert all(TRUTH[b]['work'].index.equals(reference_grid) for b in BUILDING_NAMES)
load_raw = pd.DataFrame(index=reference_grid)
SEASONS = {}
for year in sorted(set(reference_grid.year)):
    year_grid = reference_grid[reference_grid.year == year]
    if len(year_grid):
        SEASONS[f'season_{year}'] = (year_grid.min(), year_grid.max())
ANALYSIS_DATES = _analysis_dates_index()

# Identify all reference rows from the original model inference exactly once.
for b in BUILDING_NAMES:
    saved = MLP_INTERVALS[b]
    regenerated = clean_isolated_binary_flips(
        threshold_state(saved['score_mean'], CFG.decision_threshold)
    )
    np.testing.assert_array_equal(
        regenerated.fillna(-999).to_numpy(),
        pd.to_numeric(saved['predicted_state'], errors='coerce').fillna(-999).to_numpy(),
        err_msg=f'Frozen MLP prediction parity check failed for {PUBLIC_CODE[b]}',
    )
print('Validated original cached inputs and frozen five-seed predictions for B1–B7.')

## Classifier fitting

For each held-out system, select hyperparameters using the validation system only. The training systems and held-out test system follow the original LOSO assignment. The MLP predictions remain frozen.

In [ ]:
# 4. Fit logistic regression and gradient boosting; never train the MLP.

def design_matrix(frame, center, scale):
    X = frame[FEATURE_NAMES].to_numpy(dtype=np.float32)
    X = (X - np.asarray(center, np.float32)) / np.asarray(scale, np.float32)
    assert np.isfinite(X).all()
    return X


def train_and_select(model_kind, X_train, y_train, train_weight, X_val, y_val):
    candidates = LOGISTIC_GRID if model_kind == 'logistic' else BOOSTING_GRID
    fitted = []
    for configuration in candidates:
        if model_kind == 'logistic':
            model = LogisticRegression(
                C=float(configuration), penalty='l2', solver='lbfgs',
                max_iter=700, tol=1e-5,
            )
        else:
            model = HistGradientBoostingClassifier(
                **configuration, max_bins=255, early_stopping=False,
                random_state=42,
            )
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', category=FutureWarning)
            model.fit(X_train, y_train, sample_weight=train_weight)
        p_val = model.predict_proba(X_val)[:, 1]
        loss = log_loss(y_val, np.clip(p_val, 1e-10, 1 - 1e-10), labels=[0, 1])
        fitted.append((float(loss), model, configuration))
    fitted.sort(key=lambda x: x[0])
    return fitted[0]  # Validation loss, trained estimator, candidate configuration.


def check_frozen_mlp(building, frame):
    frozen = MLP_INTERVALS[building]
    p = pd.to_numeric(frozen['score_mean'], errors='coerce')
    reconstructed = clean_isolated_binary_flips(threshold_state(p, CFG.decision_threshold))
    expected = pd.to_numeric(frozen['predicted_state'], errors='coerce')
    np.testing.assert_array_equal(reconstructed.fillna(-9), expected.fillna(-9))
    assert p.index.equals(frame.index)
    return p

scores_by_model = {'MLP (frozen)': {}, 'Logistic regression': {}, 'Gradient boosting': {}}
selected_configurations = []
folds = BUILDING_NAMES if MAX_FOLDS is None else BUILDING_NAMES[:int(MAX_FOLDS)]

for test_system in folds:
    train_systems, validation_system = fold_split(test_system)
    assert len(train_systems) == 5 and validation_system != test_system
    assert test_system not in train_systems and validation_system not in train_systems

    train_df = supervised_rows(train_systems)
    validation_df = supervised_rows([validation_system])
    assert len(train_df) and len(validation_df)
    center, scale = fit_fold_scaler(train_df)
    X_train = design_matrix(train_df, center, scale)
    X_validation = design_matrix(validation_df, center, scale)
    X_test = design_matrix(FEATURE_TABLES[test_system], center, scale)
    y_train = train_df['true_state'].to_numpy(dtype=int)
    y_validation = validation_df['true_state'].to_numpy(dtype=int)
    weight = balanced_sample_weights(train_df) * len(train_df)
    assert set(np.unique(y_train)) == {0, 1}

    scores_by_model['MLP (frozen)'][test_system] = check_frozen_mlp(
        test_system, FEATURE_TABLES[test_system]
    )
    for kind, label in [('logistic', 'Logistic regression'),
                        ('boosting', 'Gradient boosting')]:
        start = time.monotonic()
        val_loss, model, selected = train_and_select(
            kind, X_train, y_train, weight, X_validation, y_validation
        )
        p_test = model.predict_proba(X_test)[:, 1]
        scores_by_model[label][test_system] = pd.Series(
            p_test, index=FEATURE_TABLES[test_system].index, dtype=float
        )
        selected_configurations.append({
            'model': label, 'system': PUBLIC_CODE[test_system],
            'validation_system': PUBLIC_CODE[validation_system],
            'n_training_rows': len(train_df), 'n_validation_rows': len(validation_df),
            'validation_log_loss': val_loss,
            'selected_parameters': json.dumps(selected, sort_keys=True),
            'fit_time_seconds': round(time.monotonic() - start, 2),
        })
    print(f'{PUBLIC_CODE[test_system]}: two baselines trained and evaluated; MLP reused.')

selected_df = pd.DataFrame(selected_configurations)
selected_df.to_csv(COMPARISON_DIR / 'baseline_validation_selection.csv', index=False)
display(selected_df)

## Interval and schedule evaluation

Apply the same fixed threshold and post-processing to each classifier. Report state classification, principal daily events, event timing, and monthly schedule-distribution errors.

In [ ]:
# 5. Exact reference metrics, daily events, and monthly schedule summaries.

def evaluate_model(model_name, system, score):
    frame = FEATURE_TABLES[system]
    frozen = MLP_INTERVALS[system]
    assert score.index.equals(frame.index)
    pred_state = clean_isolated_binary_flips(threshold_state(score, CFG.decision_threshold))
    out = pd.DataFrame({
        'true_state': frame['true_state'],
        'score_mean': score, 'predicted_state': pred_state,
    }, index=frame.index)
    original_mask = frozen['true_state'].notna() & frozen['score_mean'].notna() & frozen['predicted_state'].notna()
    baseline_mask = out['true_state'].notna() & out['score_mean'].notna() & out['predicted_state'].notna()
    assert np.array_equal(original_mask.to_numpy(), baseline_mask.to_numpy()), 'Evaluation mask differs from MLP'
    y = out.loc[baseline_mask, 'true_state'].astype(int).to_numpy()
    p = out.loc[baseline_mask, 'predicted_state'].astype(int).to_numpy()
    prob = out.loc[baseline_mask, 'score_mean'].to_numpy(float)
    row = {
        'model': model_name, 'system': PUBLIC_CODE[system], 'n_intervals': int(len(y)),
        'accuracy': accuracy_score(y, p),
        'balanced_accuracy': balanced_accuracy_score(y, p),
        'precision_on': precision_score(y, p, zero_division=0),
        'recall_on': recall_score(y, p, zero_division=0),
        'f1_on': f1_score(y, p, zero_division=0),
        'mcc': matthews_corrcoef(y, p),
        'auroc': roc_auc_score(y, prob) if len(np.unique(y)) > 1 else np.nan,
        'auprc': average_precision_score(y, prob) if len(np.unique(y)) > 1 else np.nan,
    }
    truth_binary = TRUTH[system]['work']['binary_on']
    daily = build_daily_comparison(truth_binary, pred_state)
    daily['building'] = system
    daily['model'] = model_name
    daily['system'] = PUBLIC_CODE[system]
    eligible = daily.loc[daily['daily_eval_eligible'].fillna(False)]
    for event, prefix in [('turn_on', 'startup'), ('turn_off', 'shutdown')]:
        row.update({f'{prefix}_{k.removeprefix(event + "_")}': v for k, v in
                    event_detection_stats(eligible[f'true_{event}'], eligible[f'pred_{event}'], event).items()})
        row.update({f'{prefix}_{k.removeprefix(event + "_")}': v for k, v in
                    matched_timing_stats(eligible[f'true_{event}'], eligible[f'pred_{event}'], event).items()})
    return out, daily, row


def summarize_months(model_name, system, daily):
    results = []
    source_state = TRUTH[system]['work']['binary_on']
    feat = FEATURE_TABLES[system]
    for (year, month), feature_month in feat.groupby(['year', 'month']):
        year, month = int(year), int(month)
        d = daily.loc[(daily.date.dt.year == year) & (daily.date.dt.month == month)]
        d = d.loc[d['daily_eval_eligible'].fillna(False)]
        qc = month_qc(feature_month, source_state, year, month)
        result = {'model': model_name, 'system': PUBLIC_CODE[system],
                  'month': f'{year:04d}-{month:02d}', **qc}
        for event, key in [('turn_on', 'on'), ('turn_off', 'off')]:
            truth_stats = schedule_distribution_stats(d[f'true_{event}'], 'true')
            pred_stats = schedule_distribution_stats(d[f'pred_{event}'], 'pred')
            for metric in ['median', 'iqr']:
                a, b = truth_stats[f'true_{metric}_min'], pred_stats[f'pred_{metric}_min']
                result[f'{key}_{metric}_ae_min'] = abs(a-b) if np.isfinite(a) and np.isfinite(b) else np.nan
        results.append(result)
    return results

interval_objects, daily_objects, pointwise_rows, monthly_rows = {}, {}, [], []
for model_name, by_system in scores_by_model.items():
    for system, score in by_system.items():
        interval, daily, row = evaluate_model(model_name, system, score)
        interval_objects[(model_name, system)] = interval
        daily_objects[(model_name, system)] = daily
        pointwise_rows.append(row)
        monthly_rows.extend(summarize_months(model_name, system, daily))
        if SAVE_INTERVAL_PREDICTIONS:
            interval.to_csv(COMPARISON_DIR / f'{model_name.lower().replace(" ", "_")}_{PUBLIC_CODE[system]}_intervals_PRIVATE.csv')

pointwise_df = pd.DataFrame(pointwise_rows)
monthly_df = pd.DataFrame(monthly_rows)
pointwise_df.to_csv(COMPARISON_DIR / 'state_and_event_metrics_by_system.csv', index=False)
monthly_df.to_csv(COMPARISON_DIR / 'monthly_schedule_errors.csv', index=False)

# Parity check: canonical saved MLP interval metrics must match published evaluation values.
benchmark_path = TABLE_DIR / 'pointwise_metrics_by_building.csv'
benchmark = pd.read_csv(benchmark_path)
for system in folds:
    ref = benchmark.loc[benchmark.building.eq(system)].iloc[0]
    current = pointwise_df.loc[(pointwise_df.model == 'MLP (frozen)') &
                               (pointwise_df.system == PUBLIC_CODE[system])].iloc[0]
    for metric in ['f1_on', 'mcc', 'auroc', 'auprc']:
        np.testing.assert_allclose(current[metric], ref[metric], atol=1e-8, rtol=1e-8)

cols = ['model', 'system', 'f1_on', 'mcc', 'auroc', 'auprc',
        'startup_f1', 'startup_mae_min', 'shutdown_f1', 'shutdown_mae_min']
display(pointwise_df[cols].round(3))

## Aggregate metrics and uncertainty

Compute macro and pooled results using daily clusters. The paired hierarchical bootstrap resamples systems and days jointly across methods, preserving the comparison structure.

In [ ]:
# 6. Original day-cluster aggregation and bootstrap metric definitions (unmodified).
CFG_BASE = CFG

def _confusion_counts(y, p):
    y = np.asarray(y, int); p = np.asarray(p, int)
    return {
        'tn': int(np.sum((y == 0) & (p == 0))),
        'fp': int(np.sum((y == 0) & (p == 1))),
        'fn': int(np.sum((y == 1) & (p == 0))),
        'tp': int(np.sum((y == 1) & (p == 1))),
    }

def _f1_from_counts(tn, fp, fn, tp):
    den = 2*tp + fp + fn
    return 2*tp/den if den else np.nan

def _mcc_from_counts(tn, fp, fn, tp):
    den = math.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
    return (tp*tn - fp*fn)/den if den else np.nan

def _event_prf(true_exists, pred_exists):
    t = np.asarray(true_exists, bool); p = np.asarray(pred_exists, bool)
    tp = int(np.sum(t & p)); fp = int(np.sum(~t & p)); fn = int(np.sum(t & ~p))
    precision = tp/(tp+fp) if (tp+fp) else np.nan
    recall = tp/(tp+fn) if (tp+fn) else np.nan
    f1 = 2*precision*recall/(precision+recall) if np.isfinite(precision) and np.isfinite(recall) and (precision+recall)>0 else np.nan
    return {'precision':precision,'recall':recall,'f1':f1,'tp':tp,'fp':fp,'fn':fn}

def _timing_metrics(errors):
    e = pd.to_numeric(pd.Series(errors), errors='coerce').dropna().to_numpy(float)
    if len(e) == 0:
        return {k:np.nan for k in ['mae','median_ae','rmse','bias','q90_ae','within15','within30']} | {'matched_n':0}
    ae = np.abs(e)
    return {
        'matched_n': int(len(e)),
        'mae': float(np.mean(ae)),
        'median_ae': float(np.median(ae)),
        'rmse': float(np.sqrt(np.mean(e**2))),
        'bias': float(np.mean(e)),
        'q90_ae': float(np.quantile(ae, 0.90)),
        'within15': float(np.mean(ae <= CFG_BASE.timing_tolerance_15_min)),
        'within30': float(np.mean(ae <= CFG_BASE.timing_tolerance_30_min)),
    }

def build_day_cluster_tables(interval_by_building, daily_df):
    out = {}
    for b in BUILDING_NAMES:
        pred = interval_by_building[b].copy()
        pred['date'] = pred.index.normalize()
        valid = pred['true_state'].notna() & pred['predicted_state'].notna()
        interval_rows = []
        for date, g in pred.loc[valid].groupby('date'):
            cc = _confusion_counts(g['true_state'].astype(int), g['predicted_state'].astype(int))
            interval_rows.append({'date':pd.Timestamp(date), **cc, 'n_intervals':int(len(g))})
        interval_day = pd.DataFrame(interval_rows)

        daily = daily_df.loc[daily_df['building'].eq(b)].copy()
        daily['date'] = pd.to_datetime(daily['date']).dt.normalize()
        daily['on_true_exists'] = daily['true_turn_on'].notna()
        daily['on_pred_exists'] = daily['pred_turn_on'].notna()
        daily['off_true_exists'] = daily['true_turn_off'].notna()
        daily['off_pred_exists'] = daily['pred_turn_off'].notna()
        keep = [
            'date','daily_eval_eligible',
            'on_true_exists','on_pred_exists','off_true_exists','off_pred_exists',
            'turn_on_error_min','turn_off_error_min',
        ]
        merged = daily[keep].merge(interval_day, on='date', how='outer')
        for c in ['tn','fp','fn','tp','n_intervals']:
            merged[c] = pd.to_numeric(merged[c], errors='coerce').fillna(0).astype(int)
        merged['daily_eval_eligible'] = merged['daily_eval_eligible'].fillna(False).astype(bool)
        for c in ['on_true_exists','on_pred_exists','off_true_exists','off_pred_exists']:
            merged[c] = merged[c].fillna(False).astype(bool)
        out[b] = merged.sort_values('date').reset_index(drop=True)
    return out

def compute_metrics_from_sampled_clusters(sampled_clusters):
    per_system = []
    pooled_parts = []
    for df in sampled_clusters:
        df = df.copy(); pooled_parts.append(df)
        tn,fp,fn,tp = [int(df[c].sum()) for c in ['tn','fp','fn','tp']]
        row = {
            'state_f1': _f1_from_counts(tn,fp,fn,tp),
            'state_mcc': _mcc_from_counts(tn,fp,fn,tp),
        }
        elig = df['daily_eval_eligible'].to_numpy(bool)
        for ev in ['on','off']:
            det = _event_prf(df.loc[elig,f'{ev}_true_exists'], df.loc[elig,f'{ev}_pred_exists'])
            for k in ['precision','recall','f1']:
                row[f'{ev}_event_{k}'] = det[k]
            tm = _timing_metrics(df.loc[elig,f'turn_{ev}_error_min'])
            for k,v in tm.items(): row[f'{ev}_timing_{k}'] = v
        per_system.append(row)

    pooled = pd.concat(pooled_parts, ignore_index=True)
    tn,fp,fn,tp = [int(pooled[c].sum()) for c in ['tn','fp','fn','tp']]
    result = {
        'state_f1_pooled': _f1_from_counts(tn,fp,fn,tp),
        'state_mcc_pooled': _mcc_from_counts(tn,fp,fn,tp),
        'state_f1_macro': float(np.nanmean([r['state_f1'] for r in per_system])),
        'state_mcc_macro': float(np.nanmean([r['state_mcc'] for r in per_system])),
    }
    elig = pooled['daily_eval_eligible'].to_numpy(bool)
    for ev in ['on','off']:
        det = _event_prf(pooled.loc[elig,f'{ev}_true_exists'], pooled.loc[elig,f'{ev}_pred_exists'])
        for k in ['precision','recall','f1']:
            result[f'{ev}_event_{k}_pooled'] = det[k]
            result[f'{ev}_event_{k}_macro'] = float(np.nanmean([r[f'{ev}_event_{k}'] for r in per_system]))
        tm = _timing_metrics(pooled.loc[elig,f'turn_{ev}_error_min'])
        for k in ['mae','median_ae','rmse','bias','q90_ae','within15','within30']:
            result[f'{ev}_timing_{k}_pooled'] = tm[k]
            result[f'{ev}_timing_{k}_macro'] = float(np.nanmean([r[f'{ev}_timing_{k}'] for r in per_system]))
    return result

In [ ]:
# 7. Macro/pooled metrics, matched months, and paired hierarchical bootstrap.

# We build the audited cluster objects one model at a time, maintaining the same date index.
cluster_by_model = {}
for model_name in scores_by_model:
    intervals = {b: interval_objects[(model_name, b)] for b in folds}
    daily_combined = pd.concat([daily_objects[(model_name, b)] for b in folds], ignore_index=True)
    # The original helper iterates over BUILDING_NAMES. In a one-fold smoke run, set a local list.
    all_systems = BUILDING_NAMES
    BUILDING_NAMES = list(folds)
    try:
        cluster_by_model[model_name] = build_day_cluster_tables(intervals, daily_combined)
    finally:
        BUILDING_NAMES = all_systems

summary = []
for name, groups in cluster_by_model.items():
    record = compute_metrics_from_sampled_clusters([groups[b] for b in folds])
    summary.append({'model': name, **record})
summary_df = pd.DataFrame(summary)

eligible_months = monthly_df.loc[monthly_df['primary_evaluation_eligible']].copy()
# Fair monthly comparison: restrict all three models to identical system-months.
month_key = ['system', 'month']
month_metric_cols = [f'{event}_{metric}_ae_min'
                     for event in ['on', 'off'] for metric in ['median', 'iqr']]
matched_months = {}
for col in month_metric_cols:
    valid = eligible_months.loc[eligible_months[col].notna()]
    count = valid.groupby(month_key)['model'].nunique()
    matched_months[col] = count.index[count.eq(3)]

for event in ['on', 'off']:
    for metric in ['median', 'iqr']:
        col = f'{event}_{metric}_ae_min'
        values = eligible_months.groupby('model')[col].mean()
        summary_df[f'{event}_{metric}_monthly_mae_min'] = summary_df['model'].map(values)
        counts = eligible_months.groupby('model')[col].count()
        summary_df[f'{event}_{metric}_monthly_n'] = summary_df['model'].map(counts)
        aligned = eligible_months.set_index(month_key)
        aligned = aligned.loc[aligned.index.isin(matched_months[col])]
        aligned_values = aligned.groupby('model')[col].mean()
        summary_df[f'{event}_{metric}_matched_monthly_mae_min'] = summary_df['model'].map(aligned_values)
        summary_df[f'{event}_{metric}_matched_monthly_n'] = len(matched_months[col])
summary_df.to_csv(COMPARISON_DIR / 'main_comparison_summary.csv', index=False)
main_cols = ['model', 'state_f1_macro', 'state_f1_pooled',
             'on_event_f1_pooled', 'off_event_f1_pooled',
             'on_timing_mae_pooled', 'off_timing_mae_pooled',
             'on_median_monthly_mae_min', 'off_median_monthly_mae_min']
display(summary_df[main_cols].round(3))

# Resample shared systems and shared days; compare all methods on paired draws.
paired_rows = []
if RUN_PAIRED_BOOTSTRAP:
    assert len(folds) == 7, 'Full seven-system evaluation required for the paper bootstrap'
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    comparator_names = ['Logistic regression', 'Gradient boosting']
    requested = ['state_f1_macro', 'state_f1_pooled', 'on_event_f1_pooled',
                 'off_event_f1_pooled', 'on_timing_mae_pooled', 'off_timing_mae_pooled']
    samples = {(comparison, metric): [] for comparison in comparator_names for metric in requested}
    for _ in range(BOOTSTRAP_REPLICATES):
        picked_systems = rng.choice(folds, size=len(folds), replace=True)
        row_selectors = []
        for system in picked_systems:
            n = len(cluster_by_model['MLP (frozen)'][system])
            assert all(len(cluster_by_model[k][system]) == n for k in cluster_by_model)
            row_selectors.append((system, rng.integers(0, n, size=n)))
        metrics = {}
        for name, clusters in cluster_by_model.items():
            copied = [clusters[s].iloc[idx] for s, idx in row_selectors]
            metrics[name] = compute_metrics_from_sampled_clusters(copied)
        for comparator in comparator_names:
            for metric in requested:
                a = metrics['MLP (frozen)'][metric]
                b = metrics[comparator][metric]
                samples[(comparator, metric)].append(a - b if np.isfinite(a) and np.isfinite(b) else np.nan)
    center = summary_df.set_index('model')
    for comparator in comparator_names:
        for metric in requested:
            diff = center.loc['MLP (frozen)', metric] - center.loc[comparator, metric]
            vals = pd.Series(samples[(comparator, metric)]).dropna()
            paired_rows.append({'comparison': f'MLP minus {comparator}', 'metric': metric,
                                'estimate': diff, 'ci_low': vals.quantile(.025),
                                'ci_high': vals.quantile(.975), 'valid_replicates': len(vals)})
    paired_df = pd.DataFrame(paired_rows)
    paired_df.to_csv(COMPARISON_DIR / 'paired_bootstrap_differences.csv', index=False)
    display(paired_df.round(3))

print('Comparison complete. Outputs:', COMPARISON_DIR.resolve())

## Outputs

Aggregate scores, system-level metrics, monthly errors, validation selections, and paired uncertainty estimates are saved in `baseline_comparison/` under the configured results directory. The repository contains neither the private inputs nor the generated results.

To reproduce the paper comparison, run all seven folds with the settings above and the original authorized caches. The MLP is not trained in this notebook.